# Lecture 06 companion notebook — Python: files, data frames and figures

**BIOE 610 Computational Tools · Week 6**

Every example from Lecture 06 as cells you can run, change and break — reading and writing files, drawing random samples, matplotlib, and pandas data frames. There is a matching R notebook (`Lecture06_R_dataframes_figures.ipynb`) with the same sections in the same order — keep them open side by side to compare. Needs `pandas`, `numpy`, `matplotlib`, `scipy` and `seaborn` (all in the course environment).

**How to use this notebook in Positron**

1. Open the file — Positron shows it as a notebook automatically
2. Pick a **Python interpreter** if prompted (top right)
3. Run a cell with **Shift-Enter**. Run them in order the first time; then go back and experiment — *change the numbers, break things on purpose, read the errors*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Part 1 · Reading and writing files

## 1 · Reading tabular data

`pd.read_csv()` handles comma- *and* tab-separated files — `sep="\t"` makes it tab-separated — and reads straight from a URL, so this notebook works wherever you saved it.

In [ ]:
stickle = pd.read_csv(
    "https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv",
    sep="\t")
print(stickle.shape)          # (rows, columns)
stickle.iloc[0:3, 0:5]        # first rows, first columns

## 2 · Writing tabular data

Write **derived** tables into `results/` — never overwrite anything in `data/` (raw data is read-only). `index=False`, or pandas sneaks an extra unnamed column of row numbers into your file.

In [ ]:
genes  = [c for c in stickle.columns if c.startswith("Gene")]
totals = pd.DataFrame({"Individual": stickle["Individual"],
                       "Total":      stickle[genes].sum(axis=1)})
totals.head(3)

In [ ]:
Path("results").mkdir(exist_ok=True)
totals.to_csv("results/totals.csv", index=False)
totals.to_csv("results/totals.tsv", sep="\t", index=False)
sorted(p.name for p in Path("results").iterdir())

## 3 · Saving figures to files

matplotlib's pattern is simply `plt.savefig()` — the extension picks the format. Vector formats (`.svg`, `.pdf`) scale forever; raster (`.png`) needs a resolution (`dpi=300`).

In [ ]:
plt.boxplot(totals["Total"])
plt.ylabel("Total reads per fish")
plt.savefig("results/first_figure.svg")
plt.savefig("results/first_figure.png", dpi=300)
plt.show()

sorted(p.name for p in Path("results").iterdir())

# Part 2 · Drawing samples from distributions

## 4 · Drawing random samples

In Python random numbers come from a numpy **generator**: `rng = np.random.default_rng(2026)` — the number is the seed, like `set.seed(2026)` in R. The generator's methods (`normal`, `uniform`, `poisson`, `binomial`, `integers`, `choice`) are the `r*` family.

In [ ]:
rng = np.random.default_rng(2026)
x = rng.normal(0, 10, 10000)                 # rnorm(10000, 0, 10)
y = rng.integers(1, 10001, size=10000)       # sample(1:10000, ..., replace = TRUE)
plt.scatter(x, y, s=2, alpha=0.4, color="#31688e")
plt.xlabel("x : rng.normal(0, 10)"); plt.ylabel("y : rng.integers(1, 10001)")
plt.show()

In [ ]:
# Your turn within a cell: change the mean and sd and re-run.
# What does sd = 1 vs sd = 50 do to the histogram?
plt.hist(rng.normal(0, 10, 10000), bins=40)
plt.show()

## 5 · Histograms and the honest density overlay

`density=True` puts the bars on the **density** scale, so the theoretical curve `norm.pdf()` drops straight on top — no magic scaling constants. Every `plt.` call draws on the same figure until `plt.show()`.

In [ ]:
from scipy.stats import norm

x = rng.normal(0, 100, 1000)
plt.hist(x, bins=30, density=True, color="lightgrey", edgecolor="white")
grid = np.linspace(-400, 400, 200)
plt.plot(grid, norm.pdf(grid, 0, 100), color="#440154", lw=2)
plt.xlabel("x"); plt.ylabel("density")
plt.show()

## 6 · Four distributions at once — and multi-panel figures

`plt.subplots(2, 2)` hands you the grid as an array of **axes** to draw on. Four workhorses: **normal** (measurements), **uniform** (anything equally likely), **Poisson** (counts), **binomial** (successes out of *n* tries).

In [ ]:
n = 2000
draws = {"Normal(50, 10)":    rng.normal(50, 10, n),
         "Uniform(0, 100)":   rng.uniform(0, 100, n),
         "Poisson(5)":        rng.poisson(5, n),
         "Binomial(20, 0.3)": rng.binomial(20, 0.3, n)}

fig, ax = plt.subplots(2, 2, figsize=(8, 6))
for a, (name, d), c in zip(ax.flat, draws.items(),
                           ["#440154", "#31688e", "#35b779", "#fde725"]):
    a.hist(d, bins=40, color=c, edgecolor="white")
    a.set_title(name)
fig.tight_layout()
plt.show()

In [ ]:
# Change n, the parameters and bins until each panel surprises you.
# What happens to Poisson(5) as you raise lam to 50?
plt.hist(rng.poisson(50, 2000), bins=40, color="#35b779", edgecolor="white")
plt.show()

# Part 3 · matplotlib vocabulary and data frames

## 7 · matplotlib vocabulary

matplotlib layers the same way base R does: a first `plt.plot()` starts the figure, further calls add to it, and `plt.xlabel()` / `plt.title()` / `plt.legend()` do the styling.

In [ ]:
seq_1 = np.linspace(0, 10, 101)
plt.plot(seq_1, np.sin(seq_1), lw=2, color="#440154")             # the line
plt.plot(seq_1[::10], np.sin(seq_1)[::10], "o", color="#35b779")  # every 10th point
plt.xlabel("x"); plt.ylabel("sin(x)")
plt.title("matplotlib layers the same way")
plt.legend(["line", "every 10th point"])
plt.show()

## 8 · Creating a data frame by hand

A `DataFrame` is built from a dictionary of `column name: values`. The row names are the **index**.

In [ ]:
hydrogel_concentration = pd.Categorical(["low", "high", "high", "high",
                                         "medium", "medium", "medium", "low"])
compression  = [3.4, 3.4, 8.4, 3, 5.6, 8.1, 8.3, 4.5]
conductivity = [0, 9.2, 3.8, 5, 5.6, 4.1, 7.1, 5.3]

mydata = pd.DataFrame({"hydrogel_concentration": hydrogel_concentration,
                       "compression": compression,
                       "conductivity": conductivity})
mydata.index = [f"Sample_{i}" for i in range(1, 9)]
mydata

## 9 · The stickleback table as a data frame

First questions for any new table: how big (`.shape`), what's in it (`.head()`, `.info()`), how do the groups break down (`pd.crosstab`), what does one column look like (`.describe()`).

In [ ]:
print(stickle.shape)
pd.crosstab(stickle["Genotype"], stickle["Microbiota"])

In [ ]:
stickle["Gene1"].describe()

## 10 · Summary statistics by group

`.groupby()` + an aggregation is the pandas counterpart of R's `aggregate()`; it generalizes to many statistics at once with `.agg()` (next week's `group_by() |> summarize()`).

In [ ]:
stickle.groupby(["Genotype", "Microbiota"])["Gene1"].median()

In [ ]:
stickle[["Gene1", "Gene2", "Gene3"]].mean()

## 11 · A first real figure

In [ ]:
stickle.boxplot(column="Gene1", by=["Genotype", "Microbiota"])
plt.suptitle("")
plt.title("Gene1 counts by host genotype and microbiota")
plt.ylabel("Gene1 read count")
plt.show()

## 12 · The same figure, dressed for a lab meeting

seaborn derives the grouping, colors and legend *from the data* — a preview of next week, where the grammar-of-graphics idea takes over fully.

In [ ]:
import seaborn as sns

sns.boxplot(data=stickle, x="Genotype", y="Gene1", hue="Microbiota",
            palette=["#440154", "#35b779"])
sns.stripplot(data=stickle, x="Genotype", y="Gene1", hue="Microbiota",
              dodge=True, size=4, alpha=0.6, palette="dark:black", legend=False)
plt.xlabel("Host genotype"); plt.ylabel("Gene1 read count")
plt.title("Stickleback gut RNA-seq: Gene1 by group")
plt.show()

## Your turn

1. Pick another gene (`Gene2` … `Gene600`) and redo sections 10–12 for it. Which gene shows the biggest group difference you can find?
2. Change the gallery in section 6: new parameters, new `bins`, or swap a panel for `rng.exponential(5, n)`.
3. Put it in a script: `scripts/summarize_stickleback.py` that reads the data, writes a per-group median CSV into `results/`, and saves the gallery as `results/distributions.svg` *and* a 300-dpi PNG. A working version is on the Examples page — but only look after your own runs.